# Bader Charge Analysis — CuTA / MgTA MOFs

Compares Bader charges between hydrated and dehydrated structures for the orthorhombic metal-triazolate frameworks (CuTA, MgTA). Reads VASP `POSCAR` files and Henkelman-group `ACF.dat` Bader output, computes net charge per atom (formal valence − Bader charge), and writes a summary comparison CSV per system.

**Expected input layout** (relative to the notebook):
```
data/
  cuta/
    POSCAR_hydr
    POSCAR_dehy
    ACF_hydr_CuTA.dat
    ACF_dehy_CuTA.dat
  mgta/
    POSCAR_hydr
    POSCAR_dehy
    ACF_hydr_MgTA.dat
    ACF_dehy_MgTA.dat
```
Outputs are written to `results/`.

# Packages

In [ ]:
!pip install numpy pandas matplotlib

# Setup

Shared parsing and analysis functions used for both CuTA and MgTA. Formal valences below assume standard VASP PAW pseudopotentials — verify against `ZVAL` in each system's `OUTCAR` before trusting the output.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Formal valences shared across systems; the metal is added per-call in run_bader_analysis
BASE_VALENCE = {
    'C': 4,
    'O': 6,
    'H': 1,
    'N': 5,
}

# ── Robust POSCAR parser (handles Selective dynamics line) ───
def parse_poscar(path):
    with open(path) as f:
        lines = [l.rstrip() for l in f.readlines()]

    # Start from line index 5; skip "Selective dynamics" if present
    idx = 5
    if lines[idx].strip().lower().startswith('s'):
        idx += 1

    candidate_species = lines[idx].split()
    candidate_counts  = lines[idx + 1].split()

    # If line looks like element symbols (all letters) -> VASP5 format
    if all(s.isalpha() for s in candidate_species):
        species = candidate_species
        counts  = [int(x) for x in candidate_counts]
    else:
        raise ValueError(
            f"Could not find species names in {path}.\n"
            "Your POSCAR may be VASP4 format. Add element symbols on line 6 manually."
        )

    atom_list = []
    for el, n in zip(species, counts):
        atom_list.extend([el] * n)

    print(f"\n[POSCAR] {path}")
    print(f"  Species : {species}")
    print(f"  Counts  : {counts}")
    print(f"  Total atoms: {sum(counts)}")

    return atom_list, species, counts

def parse_acf(path):
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) == 7:
                try:
                    rows.append({
                        'atom_id'     : int(parts[0]),
                        'x'           : float(parts[1]),
                        'y'           : float(parts[2]),
                        'z'           : float(parts[3]),
                        'bader_charge': float(parts[4]),
                        'min_dist'    : float(parts[5]),
                        'vol'         : float(parts[6]),
                    })
                except ValueError:
                    pass
    df = pd.DataFrame(rows)
    print(f"[ACF]    {path}  ->  {len(df)} atoms found")
    return df

def build_charge_table(acf_path, poscar_path, label, valence):
    atom_list, species, counts = parse_poscar(poscar_path)
    df = parse_acf(acf_path)

    if len(df) != len(atom_list):
        print(f"WARNING  {label}: ACF has {len(df)} atoms but POSCAR maps {len(atom_list)} -- mismatch!")
    else:
        print(f"OK  {label}: atom count matches ({len(df)})")

    df['element'] = atom_list[:len(df)]
    df['valence'] = df['element'].map(valence)

    missing = df[df['valence'].isna()]['element'].unique()
    if len(missing):
        print(f"WARNING  Elements not in valence dict: {missing} -- add them!")

    df['net_charge'] = df['valence'] - df['bader_charge']
    df['label'] = label
    return df

def summarise(df):
    return df.groupby('element')['net_charge'].agg(
        count='count', mean='mean', std='std', min='min', max='max'
    ).round(4)

def run_bader_analysis(name, metal, metal_valence, data_dir, results_dir, element_order):
    """Run the full hydrated-vs-dehydrated Bader comparison for one system
    (e.g. name='CuTA', metal='Cu', metal_valence=11)."""
    data_dir = Path(data_dir)
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    valence = {**BASE_VALENCE, metal: metal_valence}

    hydr_df  = build_charge_table(data_dir / f'ACF_hydr_{name}.dat', data_dir / 'POSCAR_hydr', 'Hydrated', valence)
    dehyd_df = build_charge_table(data_dir / f'ACF_dehy_{name}.dat', data_dir / 'POSCAR_dehy', 'Dehydrated', valence)

    sum_hydr  = summarise(hydr_df)
    sum_dehyd = summarise(dehyd_df)

    common = sum_hydr.index.intersection(sum_dehyd.index)
    comparison = pd.concat([
        sum_dehyd.loc[common, 'mean'].rename('Dehydrated (e)'),
        sum_hydr .loc[common, 'mean'].rename('Hydrated (e)'),
        (sum_hydr.loc[common, 'mean'] - sum_dehyd.loc[common, 'mean']).rename('Delta (Hydr - Dehyd)'),
    ], axis=1).round(4)

    print(f"\n=== {name}: Delta positive -> more cationic after H2O adsorption ===\n")
    print(comparison.to_string())

    print("\n" + "="*65)
    print(f"  {name}: per-atom charge detail")
    print("="*65)

    for el in element_order:
        dh = dehyd_df[dehyd_df.element == el][['atom_id', 'bader_charge', 'net_charge']]
        hy = hydr_df [hydr_df.element  == el][['atom_id', 'bader_charge', 'net_charge']]
        merged = dh.merge(hy, on='atom_id', suffixes=('_dehyd', '_hydr'), how='inner')
        if merged.empty:
            continue
        merged['Delta net_charge'] = (merged['net_charge_hydr'] - merged['net_charge_dehyd']).round(4)
        print(f"\n--- {el} ({len(merged)} atoms matched) ---")
        print(merged.round(4).to_string(index=False))

    for el, lbl in [('O', 'water O'), ('H', 'water H')]:
        dehyd_ids = set(dehyd_df[dehyd_df.element == el]['atom_id'])
        extra = hydr_df[(hydr_df.element == el) & (~hydr_df['atom_id'].isin(dehyd_ids))]
        if not extra.empty:
            print(f"\n--- {lbl} unique to hydrated ({len(extra)} atoms) ---")
            print(extra[['atom_id', 'bader_charge', 'net_charge']].round(4).to_string(index=False))

    out_path = results_dir / f'{name.lower()}_charges_comparison.csv'
    comparison.to_csv(out_path)
    print(f"\nSaved: {out_path}")

    return comparison, hydr_df, dehyd_df

# CuTA

In [ ]:
cuta_comparison, cuta_hydr_df, cuta_dehyd_df = run_bader_analysis(
    name='CuTA',
    metal='Cu',
    metal_valence=11,  # verify with: grep "ZVAL" OUTCAR
    data_dir='data/cuta',
    results_dir='results',
    element_order=['Cu', 'C', 'O', 'N', 'H'],
)

# MgTA

In [ ]:
mgta_comparison, mgta_hydr_df, mgta_dehyd_df = run_bader_analysis(
    name='MgTA',
    metal='Mg',
    metal_valence=2,  # verify with: grep "ZVAL" OUTCAR
    data_dir='data/mgta',
    results_dir='results',
    element_order=['Mg', 'C', 'O', 'N', 'H'],
)